# Grain, facts and dimensions

Core stage, lesson C17. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** state the grain of a table in one sentence, keep the finest grain the questions need, tell a measure from a description and a key, and count orders at the grain of a line without counting lines.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the cafe-sales data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it. The data is synthetic: a script made it from stated rules.

## Setup

In [ ]:
import decimal
import math
import numbers
import subprocess
import sys

try:
    import duckdb
except ImportError:  # Colab usually has DuckDB; install it when it does not
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
    import duckdb

# Run this cell as it is. It also holds the answer checker and the helpers of the exercise cells.
# Run all from the top: restart the session, run every cell from the first, then read the last cell.
EXERCISES = ['Exercise 1', 'Exercise 2', 'Exercise 3']


RESULTS = {}


def same(got, expected):
    """True when two values agree: floats within rounding, lists and dicts item by item."""
    try:
        if isinstance(got, decimal.Decimal):
            got = int(got) if got == got.to_integral_value() else float(got)
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k]) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=1e-9)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps.items() if isinstance(traps, dict) else (traps or [])):
        if same(got, wrong):
            print("Hint:", message)
    return False


def check_all(label, known):
    """Compare several variables with their known answers: name -> (answer, tolerance). It never stops Run all."""
    got = {name: globals().get(name) for name in known}
    if all(value is None for value in got.values()):
        RESULTS[label] = None
        print(f"{label}: give " + ", ".join(known) + " a value, then run this cell again.")
        return False
    wrong = [name for name, (want, tol) in known.items() if got[name] is None or not same(got[name], want)]
    RESULTS[label] = not wrong
    if wrong:
        print(f"{label}: not yet. Check " + ", ".join(wrong) + ".")
    else:
        print(f"{label}: correct.")
    return not wrong


def show(sql, limit=12):
    """Run a statement and print its rows as text: the column names, then one line for each row."""
    cur = con.execute(sql)
    if cur.description is None:
        print("(the statement ran and returns no rows)")
        return
    rows = cur.fetchall()
    print(" | ".join(column[0] for column in cur.description))
    for row in rows[:limit]:
        print(" | ".join("NULL" if value is None else str(value) for value in row))
    if len(rows) > limit:
        print(f"... {len(rows)} rows in all")


def rows(sql):
    """Run a query and return its rows as lists, with decimals as numbers and dates as text."""
    cur = con.execute(sql)
    return [[float(v) if isinstance(v, decimal.Decimal) else (str(v) if hasattr(v, "isoformat") else v) for v in row] for row in cur.fetchall()]


def say_error(sql):
    """Run a statement; when the database refuses it, print the first line of its message instead of stopping."""
    try:
        show(sql)
    except duckdb.Error as error:
        print("Refused:", str(error).splitlines()[0])


def key_listing(table):
    """The primary and foreign keys of a table, as [type, column, position] lists (the query of unit 3 of C16)."""
    return [list(r) for r in con.execute(KEYS, [table]).fetchall()]


KEYS = """SELECT tc.constraint_type, kcu.column_name, kcu.ordinal_position AS position
FROM information_schema.table_constraints AS tc
JOIN information_schema.key_column_usage AS kcu
  ON kcu.constraint_name = tc.constraint_name AND kcu.table_name = tc.table_name
WHERE tc.table_name = ? AND tc.constraint_type IN ('PRIMARY KEY', 'FOREIGN KEY')
ORDER BY tc.constraint_type, kcu.ordinal_position, kcu.column_name"""

## Data

In [ ]:
# The five cafe-sales tables (version 1), read from the release tag v0.2.0, or from the repository's own copy of the same files when the network is not available.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.2.0/datasets/cafe-sales/v1/"
LOCAL = "../../datasets/cafe-sales/v1/"      # the same files, when this notebook runs inside a clone of the repository
TABLES = {                                   # the columns and types of datafromzero's cafe-sales.sql (no keys or constraints there)
    "stores": "'INTEGER','VARCHAR','VARCHAR','DATE'",
    "customers": "'INTEGER','DATE','VARCHAR','VARCHAR','INTEGER'",
    "products": "'INTEGER','VARCHAR','VARCHAR','DECIMAL(6,2)'",
    "orders": "'INTEGER','INTEGER','INTEGER','TIMESTAMP','VARCHAR','VARCHAR'",
    "order_items": "'INTEGER','INTEGER','INTEGER','INTEGER','DECIMAL(6,2)'",
}


def fetch(name):
    """The path of one CSV file: downloaded from the tag, or the local copy."""
    import os
    import tempfile
    import urllib.request
    try:
        path = os.path.join(tempfile.mkdtemp(), name + ".csv")
        urllib.request.urlretrieve(BASE + name + ".csv", path)
        return path
    except Exception:
        return LOCAL + name + ".csv"


con = duckdb.connect()                       # a new, empty database every time this cell runs
for name, types in TABLES.items():
    con.execute(f"CREATE TABLE {name} AS FROM read_csv('{fetch(name)}', types=[{types}])")

# Known answer from the dataset README: 3, 150, 16, 1,800 and 3,287 rows.
counts = [con.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0] for name in TABLES]
assert counts == [3, 150, 16, 1800, 3287]
print("Data loaded: 3 stores, 150 customers, 16 products, 1,800 orders, 3,287 order lines.")

## Unit 1: Choose the grain

This cell builds the sales at three grains: one row for each line, for each order and for each cafe and month. The second and third are added up from the first.

In [ ]:
con.execute("""CREATE OR REPLACE TABLE sales_line AS
SELECT i.order_item_id, i.order_id, o.store_id, CAST(o.ordered_at AS DATE) AS order_date,
       i.product_id, i.quantity,
       i.quantity * p.unit_price_usd - i.discount_usd AS amount_usd
FROM order_items AS i
JOIN orders AS o ON o.order_id = i.order_id
JOIN products AS p ON p.product_id = i.product_id;

CREATE OR REPLACE TABLE sales_order AS
SELECT order_id, store_id, order_date, COUNT(*) AS lines, SUM(quantity) AS units, SUM(amount_usd) AS amount_usd
FROM sales_line
GROUP BY order_id, store_id, order_date;

CREATE OR REPLACE TABLE sales_store_month AS
SELECT store_id, month(order_date) AS month, COUNT(*) AS orders, SUM(units) AS units, SUM(amount_usd) AS amount_usd
FROM sales_order
GROUP BY store_id, month(order_date)""")
print("Three tables built.")

**Predict first.** how many rows will each of the three tables have?

In [ ]:
show("""SELECT 'line' AS grain, COUNT(*) AS rows, ROUND(SUM(amount_usd), 2) AS sales FROM sales_line
UNION ALL SELECT 'order', COUNT(*), ROUND(SUM(amount_usd), 2) FROM sales_order
UNION ALL SELECT 'store-month', COUNT(*), ROUND(SUM(amount_usd), 2) FROM sales_store_month""")

**Predict first.** will the most common pair of products be two drinks, or a drink with something to eat?

In [ ]:
show("""SELECT pa.product_name AS first_product, pb.product_name AS second_product, COUNT(*) AS orders
FROM sales_line AS a
JOIN sales_line AS b ON b.order_id = a.order_id AND b.product_id > a.product_id
JOIN products AS pa ON pa.product_id = a.product_id
JOIN products AS pb ON pb.product_id = b.product_id
GROUP BY pa.product_name, pb.product_name
ORDER BY orders DESC, first_product, second_product
LIMIT 3""")

## Unit 2: Measures and descriptions

**Predict first.** which of the numbers is the sales of the cafes, and what does the sum of the prices measure?

In [ ]:
show("""SELECT ROUND(SUM(p.unit_price_usd), 2) AS sum_of_prices,
       SUM(i.quantity) AS units,
       ROUND(SUM(i.quantity * p.unit_price_usd), 2) AS gross,
       ROUND(SUM(i.discount_usd), 2) AS discounts,
       ROUND(SUM(i.quantity * p.unit_price_usd - i.discount_usd), 2) AS sales
FROM order_items AS i
JOIN products AS p ON p.product_id = i.product_id""")

## Unit 3: Counting events at the grain of a line

**Predict first.** `COUNT(*)` here counts rows of the join. Will it be close to the number of mobile orders, or well above it?

In [ ]:
show("""SELECT COUNT(*) AS mobile_lines,
       COUNT(DISTINCT i.order_id) AS mobile_orders
FROM order_items AS i
JOIN orders AS o ON o.order_id = i.order_id
WHERE o.payment_method = 'mobile'""")

## Finish it

### Exercise 1: Finish it

**Task:** A fourth grain: one row for each product and day. Finish the table `sales_product_day` from `sales_line` (unit 1 builds it). The starter stops with an error: its `GROUP BY` does not say yet that a row stands for one product and one day. Fix it, then set `answer_1` to the rows of a query that checks the table: its number of rows, its units and its sales, which must equal the units and the sales of `sales_line`.

**Expected:** `answer_1` is [rows, units, sales], sales rounded to 2 decimals.

In [ ]:
say_error("""CREATE OR REPLACE TABLE sales_product_day AS
SELECT product_id, order_date, SUM(quantity) AS units, SUM(amount_usd) AS amount_usd
FROM sales_line
GROUP BY product_id""")      # grain: one row for each product and day; the starter stops with an error
answer_1 = None  # replace None with [rows, units, sales]: the first row of rows("SELECT ... FROM sales_product_day")

In [ ]:
_ = check("Exercise 1", "answer_1", [1819, 5162, 22754.25], traps=[([3287, 5162, 22754.25], "That is the line table. The GROUP BY must list both columns of the grain: the product and the day.")])

<details>
<summary>Hint</summary>

The error names the column that is not in the `GROUP BY`. The columns of the `GROUP BY` are the grain.

</details>

<details>
<summary>Half-solution</summary>

```python
GROUP BY product_id, ...

answer_1 = rows("SELECT COUNT(*), SUM(units), ROUND(SUM(...), 2) FROM sales_product_day")[0]
```

</details>

<details>
<summary>Solution</summary>

```python
say_error("""CREATE OR REPLACE TABLE sales_product_day AS
SELECT product_id, order_date, SUM(quantity) AS units, SUM(amount_usd) AS amount_usd
FROM sales_line
GROUP BY product_id, order_date""")
answer_1 = rows("SELECT COUNT(*), SUM(units), ROUND(SUM(amount_usd), 2) FROM sales_product_day")[0]
```

1,819 rows, 5,162 units and 22,754.25 dollars: fewer rows than the 3,287 lines, the same units and the same money. A `GROUP BY` that lists exactly the columns of the grain makes every row stand for one product and one day.

</details>

## On your own

### Exercise 2: On your own

**Task:** How many orders hold both a coffee and a bakery item? Before you start, say what the result looks like: one whole number. The categories are in `products`; each order is a group of lines. Write the code in the empty cell, and set `answer_2` to the number.

**Expected:** `answer_2` is a whole number.

In [ ]:
# Exercise 2: write your code here: answer_2 = rows(""" ... """)[0][0]

In [ ]:
_ = check("Exercise 2", "answer_2", 352, traps={1259: "That counts the orders that hold a coffee, whether or not they hold a bakery item.", 648: "That counts the orders that hold a bakery item, whether or not they hold a coffee."})

<details>
<summary>Hint</summary>

Group the lines by order. For each order, count its coffee lines and its bakery lines with `COUNT(*) FILTER (WHERE ...)`, and keep the orders where both counts are above zero. Then count those orders.

</details>

<details>
<summary>Half-solution</summary>

Expected output in words: *"A number between 300 and 400."*

```python
answer_2 = rows("""SELECT COUNT(*) AS orders
FROM (SELECT i.order_id
      FROM order_items AS i
      JOIN products AS p ON p.product_id = i.product_id
      GROUP BY i.order_id
      HAVING ...)""")[0][0]
```

</details>

<details>
<summary>Solution</summary>

```python
answer_2 = rows("""SELECT COUNT(*) AS orders
FROM (SELECT i.order_id
      FROM order_items AS i
      JOIN products AS p ON p.product_id = i.product_id
      GROUP BY i.order_id
      HAVING COUNT(*) FILTER (WHERE p.category = 'coffee') > 0
         AND COUNT(*) FILTER (WHERE p.category = 'bakery') > 0)""")[0][0]
```

352 of the 1,800 orders. Only the line grain knows which products an order holds; a table with one row for each order could not answer it. Counting the lines of a coffee and a bakery product without grouping by order would count lines, not orders.

</details>

## Check yourself

Without looking back: a table has one row for each cafe and month. Name one question it can answer and one it cannot. Then try the exercise.

<details>
<summary>answer</summary>

It can give the sales of a cafe in a month. It cannot give the sales of a product, or of a day, because that detail was added up and cannot be split back.

</details>

### Exercise 3: Check yourself: spot the flaw

**Task:** Spot the flaw. A colleague reports the customers who ordered in the second quarter (April to June) by counting the customers of each month and adding the three counts. Fix the query so that `answer_3` is the number of customers who ordered in the second quarter.

**Expected:** `answer_3` is a whole number.

In [ ]:
answer_3 = rows("""SELECT SUM(customers) AS customers_in_q2
FROM (SELECT month(ordered_at) AS month, COUNT(DISTINCT customer_id) AS customers
      FROM orders
      WHERE ordered_at >= TIMESTAMP '2026-04-01'
      GROUP BY month(ordered_at))""")[0][0]      # colleague: the three monthly counts, added

In [ ]:
_ = check("Exercise 3", "answer_3", 144, traps={370: "That adds the monthly counts, so a customer who ordered in two months is counted twice. Count the distinct customers over the whole quarter."})

<details>
<summary>Hint</summary>

A customer who ordered in April and again in May is in the April count and in the May count. What does the sum do with that customer?

</details>

<details>
<summary>Half-solution</summary>

```python
answer_3 = rows("""SELECT COUNT(DISTINCT customer_id) AS customers_in_q2
FROM orders
WHERE ...""")[0][0]
```

</details>

<details>
<summary>Solution</summary>

```python
answer_3 = rows("""SELECT COUNT(DISTINCT customer_id) AS customers_in_q2
FROM orders
WHERE ordered_at >= TIMESTAMP '2026-04-01'""")[0][0]
```

The colleague got 370, but only 144 customers ordered in the quarter (all statuses). A count of distinct customers does not add up from months to a quarter: a customer who came in two months is counted twice in the sum. Units and sales add; a distinct count has to be made again at the grain you want, from the rows below it. The monthly table is the wrong source for a question about the quarter.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")